In [ ]:
# with open('config.json','w') as file:
#     json.dump({'login_data':login_data,'cookies':cookies,'headers':headers},file,indent=2)

In [1]:
import httpx
from bs4 import BeautifulSoup
import re
import time
import json
import logging
import sys

def configure_logging(level=logging.INFO):
    logging.basicConfig(filename='logs.log',
                        level=level,
                        format='%(asctime)s %(levelname)s: %(message)s')
    root=logging.getLogger()

    handler=logging.StreamHandler()
    handler.setLevel(level)

    root.addHandler(handler)

configure_logging()
logger=logging.getLogger(__name__)


In [ ]:

class RoyalRoadConfigError(Exception):
    """Missing config."""
    pass

def clean_spaces(string):
    string=string.strip()
    string=re.sub(r'\s+',' ',string)
    return string

class RRScrap:
    def __init__(self):
        self.base_url='https://www.royalroad.com'

        with open('config.json') as f:
            config=json.load(f)
        
        self.login_data = config['login_data']
        self.cookies = config['cookies']
        self.headers = config['headers']
         
    def load_page(self, link,timeout=15,new_base=None):
        if new_base:
            base_url=new_base
        else:
            base_url=self.base_url
        link=base_url+link
        if not hasattr(self,'client'):
            logger.warning('Auth was not done manually, using defaul parameters')
            self.auth()
        max_retries = 5
        base_delay=1
        for tries in range(1,max_retries+1):
            try:
                page=self.client.get(link,timeout=timeout)
                logger.debug(f'loaded:{link}')
                if tries>1:
                    logger.info(f'Successfully loaded:{link} after {tries} retries')
                return page
            except httpx.TimeoutException as e:
                delay=base_delay*(2**tries)
                logger.warning(f'''timeout excepcion for:{link} 
                               \n retrying in {delay} , attempt {tries}/{max_retries}''')
                time.sleep(delay)
            except httpx.HTTPStatusError as e:
                delay=base_delay*(2**tries)
                if e.response.status_code in [429, 500, 502, 503, 504]:
                    logger.warning(f'''error status code: {e.response.status_code} link:{link} \n 
                                   retrying in {delay} , attempt {tries}/{max_retries}''')
                    time.sleep(delay)
                else:
                    logger.error(f'error status code: {e.response.status_code} link:{link}')
                    raise
            except httpx.RequestError as e:
                delay=base_delay*(2**tries)
                logger.warning(f'''error during request to link:{link}\n
                               retrying in {delay} , attempt {tries}/{max_retries}''')
                time.sleep(delay)
            except Exception as e:
                logger.error(f'unexpected error loading: {link}')
                raise
        raise RuntimeError(f'Failed to load {link} after {tries}/{max_retries} attempts')
    def load_titles(self,page):
        page=BeautifulSoup(page)
        page_dic={}
        for el in page.find_all('h2',class_="fiction-title"):
            title=clean_spaces(el.get_text())
            link=el.a.get('href')
            page_dic[title]={'link':link}
        return page_dic
    def load_stat_meta(self,soup,metadata):
        span=soup.find_all('li',class_='bold uppercase')
        for x in span:
            name=x.text.replace(':','').strip()
            value=x.find_next_sibling('li')
            if value:
                value=int(value.text.strip().replace(',',''))

                metadata[f'{name}']=value
    def load_stars(self,soup,metadata):
        span=soup.find_all('span',attrs={'data-original-title':True})
        for x in span:
            name=x.get('data-original-title')
            value=x.get('aria-label') 
            value=float(re.match(r'\d+.?\d*',value).group())
            metadata[name]=value
    def load_meta(self,page):
            
        metadata={}

        soup=BeautifulSoup(page.text)

        # tags
        tags=soup.find_all('a',href=lambda x: x and 'tagsAdd=' in x)
        for i,tag in enumerate(tags):
            tags[i]=tag.text
        metadata['tags']=tags

        self.load_stars(soup,metadata)

        self.load_stat_meta(soup,metadata)

        return metadata
    def load_titles_links_meta(self,link):
        page1 = self.load_page(link)
        soup = BeautifulSoup(page1.text, 'html.parser')

        last_page_link=soup.find_all('ul',class_='pagination justify-content-center')[0].find_all('li')[-1].a.get('href') 
        self.total_pages = int(re.search(r'\d+',last_page_link).group() )
    #мог бы просто лишний раз вызвать страницу, 
    #но предпочел дополнительно обработать первую страницу, чтобы не увеличивать количество вызовов
        logger.info(f'Number of pages: {self.total_pages}')

        all_titles={}

        loaded_titles=self.load_titles(page1)
        all_titles=all_titles | loaded_titles
        logger.info(f'Loaded {len(loaded_titles)} titles, page 1')

        all_titles = all_titles | loaded_titles
        for x in range(2,self.total_pages +1):
            print(f'{link}?page={x}')
            page=self.load_page(f'{link}?page={x}')
            loaded_titles=self.load_titles(page)
            all_titles=all_titles | loaded_titles
            print(f'loaded_titles: {loaded_titles}')
            print(f'len of all_titles: {len(all_titles)}')

            logger.info(f'Loaded {len(loaded_titles)} titles, page {x}')

            time.sleep(1)
        
        for name, dic in all_titles.items():
            link=dic['link']
            dic.update(self.load_meta(self.load_page(link)))
            logger.info(f'Succesfully loaded metadata of:{name} ')
            time.sleep(1)

        logger.info(f'Succesfully loaded all {len(dic)} titles')
        return all_titles
    def close_client(self):
        if self.client:
            self.client.close_client()
            self.client = None
            print("Client connection closed.")

scrap=RRScrap()

In [24]:
all=scrap.load_titles_links_meta('/my/readlater')
all

Auth was not done manually, using defaul parameters
HTTP Request: POST https://www.royalroad.com/account/login?returnurl=%2Fhome "HTTP/1.1 302 Found"
HTTP Request: GET https://www.royalroad.com/account/loginsuccess?returnUrl=%2Fhome "HTTP/1.1 302 Found"
HTTP Request: GET https://www.royalroad.com/home "HTTP/1.1 200 OK"
Succussfully connected client
HTTP Request: GET https://www.royalroad.com/my/readlater "HTTP/1.1 200 OK"
Number of pages: 3
Loaded 50 titles, page 1


/my/readlater?page=2


HTTP Request: GET https://www.royalroad.com/my/readlater?page=2 "HTTP/1.1 200 OK"
Loaded 50 titles, page 2


loaded_titles: {'Queen of Conquest': {'link': '/fiction/85474/queen-of-conquest'}, 'Cultivating Chai [Book 2 Ongoing!]': {'link': '/fiction/87416/cultivating-chai-book-2-ongoing'}, 'Leave No Survivors [RRCM Winner Jan. 2024]': {'link': '/fiction/79558/leave-no-survivors-rrcm-winner-jan-2024'}, 'Ember of Invention': {'link': '/fiction/82942/ember-of-invention'}, 'Conquest of Avalon': {'link': '/fiction/42560/conquest-of-avalon'}, 'Markets and Multiverses (A Serial Transmigration LitRPG)': {'link': '/fiction/61244/markets-and-multiverses-a-serial-transmigration'}, 'The Acts of Androkles': {'link': '/fiction/25012/the-acts-of-androkles'}, 'The Halcyon System [Stubbed]': {'link': '/fiction/86614/the-halcyon-system-stubbed'}, 'The Flower That Bloomed Nowhere': {'link': '/fiction/28806/the-flower-that-bloomed-nowhere'}, 'The Heir Apparent [Reincarnation LitRPG]': {'link': '/fiction/84498/the-heir-apparent-reincarnation-litrpg'}, 'The Grand Weave': {'link': '/fiction/57427/the-grand-weave'}, 

HTTP Request: GET https://www.royalroad.com/my/readlater?page=3 "HTTP/1.1 200 OK"
Loaded 15 titles, page 3


loaded_titles: {"A Sinner's Eden": {'link': '/fiction/45384/a-sinners-eden'}, 'The Mine Lord: A Dwarven Survival Base-Builder': {'link': '/fiction/76164/the-mine-lord-a-dwarven-survival-base-builder'}, 'Regressor Sect Master': {'link': '/fiction/76389/regressor-sect-master'}, 'The Elder Lands (A Kingdom Building LitRPG)': {'link': '/fiction/64948/the-elder-lands-a-kingdom-building-litrpg'}, 'Otherworldly - Between the Dusk and Dawn (Books 1-3 UNSTUBBING on 9-27-2026!!)': {'link': '/fiction/46934/otherworldly-between-the-dusk-and-dawn-books-1-3'}, 'Chaotic Craftsman Worships The Cube': {'link': '/fiction/41656/chaotic-craftsman-worships-the-cube'}, 'Industrial Strength Magic': {'link': '/fiction/57011/industrial-strength-magic'}, 'Apocalypse: Reborn As A Monster (Book 2 Completed)': {'link': '/fiction/66453/apocalypse-reborn-as-a-monster-book-2-completed'}, 'Focused Fire (ATLA Fanfic)': {'link': '/fiction/73440/focused-fire-atla-fanfic'}, 'The Number': {'link': '/fiction/48012/the-numbe

HTTP Request: GET https://www.royalroad.com/fiction/94872/the-augments-code-a-superhero-litrpg-book-two "HTTP/1.1 200 OK"
Succesfully loaded metadata of:The Augment's Code (A Superhero LitRPG) [Book Two Stubbed] 
HTTP Request: GET https://www.royalroad.com/fiction/120579/blessed-a-dark-arcanepunk-litrpg-stubbed "HTTP/1.1 200 OK"
Succesfully loaded metadata of:Blessed - A Dark Arcanepunk LitRPG [Stubbed] 


KeyboardInterrupt: 

In [7]:
all

{"The Augment's Code (A Superhero LitRPG) [Book Two Stubbed]": {'link': '/fiction/94872/the-augments-code-a-superhero-litrpg-book-two'},
 'Blessed - A Dark Arcanepunk LitRPG [Stubbed]': {'link': '/fiction/120579/blessed-a-dark-arcanepunk-litrpg-stubbed'},
 'Flesh Eater: Demon Evolution LitRPG [BOOK 1 COMPLETE]': {'link': '/fiction/129187/flesh-eater-demon-evolution-litrpg-book-1-complete'},
 'Engine of Reincarnation [A Serial Rebirth Isekai LitRPG]': {'link': '/fiction/143293/engine-of-reincarnation-a-serial-rebirth-isekai'},
 'Second Life as a Soldier [Book 1 Complete]': {'link': '/fiction/126560/second-life-as-a-soldier-book-1-complete'},
 'God of Trash [Cultivation LitRPG] From Trash-Tier to the Ultimate Trash! [Five Full Books! 5!]': {'link': '/fiction/107252/god-of-trash-cultivation-litrpg-from-trash-tier'},
 'Troll in the Dungeon! (Harry Potter/OC-Insert)': {'link': '/fiction/72378/troll-in-the-dungeon-harry-potteroc-insert'},
 'Immovable Mage [Progression Fantasy]': {'link': '/f

In [5]:
scrap.client

In [21]:
with open('html.txt','w') as f:
    f.write(soup.prettify())

In [41]:
span=soup.find_all('span',attrs={'data-original-title':True})
for x in span:
    meta[x.get('data-original-title')]=x.get('aria-label')

In [40]:
span=soup.find_all('span',attrs={'data-original-title':True})
span[0].get('aria-label')

'4.42 stars'

In [53]:
float(re.match(r'\d+\.?\d*',meta['Story Score']).group())

4.56

{'Total Views': '725,476',
 'Average Views': '4,478',
 'Followers': '2,604',
 'Favorites': '703',
 'Ratings': '457',
 'Pages': '1,933'}

In [19]:
meta={}
scrap.load_stat_meta(BeautifulSoup(scrap.load_page('/fiction/94872/the-augments-code-a-superhero-litrpg-book-two')),'Total Views',meta)

HTTP Request: GET https://www.royalroad.com/fiction/94872/the-augments-code-a-superhero-litrpg-book-two "HTTP/1.1 200 OK"


In [11]:
read_later

{"The Augment's Code (A Superhero LitRPG) [Book Two Stubbing July 24th!]": '/fiction/94872/the-augments-code-a-superhero-litrpg-book-two',
 'Blessed - A Dark Arcanepunk LitRPG [Stubbed]': '/fiction/120579/blessed-a-dark-arcanepunk-litrpg-stubbed',
 'Flesh Eater: Demon Evolution LitRPG [BOOK 1 COMPLETE]': '/fiction/129187/flesh-eater-demon-evolution-litrpg-book-1-complete',
 'Engine of Reincarnation [A Serial Rebirth Isekai LitRPG]': '/fiction/143293/engine-of-reincarnation-a-serial-rebirth-isekai',
 'Second Life as a Soldier [Book 1 Complete]': '/fiction/126560/second-life-as-a-soldier-book-1-complete',
 'God of Trash [Cultivation LitRPG] From Trash-Tier to the Ultimate Trash! [Five Full Books! 5!]': '/fiction/107252/god-of-trash-cultivation-litrpg-from-trash-tier',
 'Troll in the Dungeon! (Harry Potter/OC-Insert)': '/fiction/72378/troll-in-the-dungeon-harry-potteroc-insert',
 'Immovable Mage [Progression Fantasy]': '/fiction/39344/immovable-mage-progression-fantasy',
 'Never Die Twice

In [26]:
soup=BeautifulSoup(page.text)

In [20]:
matches=[line.strip() for line in soup.text if 'ta' in line]
print(matches)

[]


In [29]:
soup.find_all('div',class_='flex-1')

[]

In [21]:
elements = soup.find_all(string=lambda text: text and 'tag' in text)
for el in elements:
    print(el)

 Global site tag (gtag.js) - Google Analytics 
        
        function createGa() {
            function adBlockEnabled() {
                var ad = document.createElement('ins');
                ad.className = 'AdSense';
                ad.style.display = 'block';
                ad.style.position = 'absolute';
                ad.style.top = '-1px';
                ad.style.height = '1px';
                document.body.appendChild(ad);
                var isAdBlockEnabled = !ad.clientHeight;
                document.body.removeChild(ad);
                return isAdBlockEnabled;
            }
        
            window.dataLayer = window.dataLayer || [];
            function gtag(){dataLayer.push(arguments);}
            gtag('js', new Date());
            gtag('set', {
                "dimension4": adBlockEnabled()
            });
    
            gtag('set', {"dimension2":"v2","dimension5":"No","dimension6":"NitroPay","user_id":"688ea5fa0eeca6097c637c9dea4c60b7"});
            fun